In [4]:
import pandas as pd 

df = pd.read_csv("../data/raw/results.csv")

print(df.head())
print(df.info())


         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49453 non-null  float64
 4   away_score  49453 non-null 

In [5]:
df.isnull().sum()


date           0
home_team      0
away_team      0
home_score    24
away_score    24
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [6]:
df[df['home_score'].isnull()]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49453,2026-06-24,Mexico,Czech Republic,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49454,2026-06-24,South Africa,South Korea,NaN,NaN,FIFA World Cup,Guadalupe,Mexico,True
49455,2026-06-24,Canada,Switzerland,NaN,NaN,FIFA World Cup,Vancouver,Canada,False
49456,2026-06-24,Bosnia and Herzegovina,Qatar,NaN,NaN,FIFA World Cup,Seattle,United States,True
49457,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49458,2026-06-24,Morocco,Haiti,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49459,2026-06-25,United States,Turkey,NaN,NaN,FIFA World Cup,Inglewood,United States,False
49460,2026-06-25,Paraguay,Australia,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
49461,2026-06-25,Curaçao,Ivory Coast,NaN,NaN,FIFA World Cup,Philadelphia,United States,True
49462,2026-06-25,Ecuador,Germany,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


In [7]:
df["tournament"].value_counts()

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
                                        ...  
Copa Confraternidad                         1
Benedikt Fontana Cup                        1
ConIFA Challenger Cup                       1
CONIFA World Cup qualification              1
South Asian Super Cup                       1
Name: count, Length: 200, dtype: int64

In [8]:
print(df["date"].min())
print(df["date"].max())

1872-11-30
2026-06-27


In [9]:
df_clean = df.copy()

In [10]:
print(df_clean["date"].dtype)
df_clean["date"] = pd.to_datetime(df_clean["date"])
print(df_clean["date"].dtype)

str
datetime64[us]


In [11]:
df_clean = df_clean[df_clean["home_score"].notna()]
df_clean.isnull().sum()
print(df_clean.shape)

(49453, 9)


In [12]:
df_clean.duplicated().sum()

np.int64(0)

In [13]:
df_clean["date"].dt.year.value_counts().sort_index()


date
1872       1
1873       1
1874       1
1875       1
1876       2
        ... 
2022     970
2023    1054
2024    1231
2025    1002
2026     359
Name: count, Length: 155, dtype: int64

In [14]:
def get_results(row):
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    elif row["home_score"] < row["away_score"]:
        return "Away Win"
    else:
        return "Draw"
    
df_clean["result"] = df_clean.apply(get_results, axis=1)

In [15]:
df_clean[["home_score", "away_score", "result"]].head(10)

,home_score,away_score,result
0,0.0,0.0,Draw
1,4.0,2.0,Home Win
2,2.0,1.0,Home Win
3,2.0,2.0,Draw
4,3.0,0.0,Home Win
5,4.0,0.0,Home Win
6,1.0,3.0,Away Win
7,0.0,2.0,Away Win
8,7.0,2.0,Home Win
9,9.0,0.0,Home Win


In [16]:
df_clean["result"].value_counts()

result
Home Win    24237
Away Win    13969
Draw        11247
Name: count, dtype: int64

In [17]:
df_clean.to_csv("../data/processed/clean_matches.csv", index=False)

In [18]:
df_clean[["home_score", "away_score", "result"]].head()

,home_score,away_score,result
0,0.0,0.0,Draw
1,4.0,2.0,Home Win
2,2.0,1.0,Home Win
3,2.0,2.0,Draw
4,3.0,0.0,Home Win


In [19]:
df_features = df_clean.copy()

df_features["year"] = df_features["date"].dt.year
df_features["month"] = df_features["date"].dt.month

df_features[["date", "year", "month"]].head()

,date,year,month
0,1872-11-30,1872,11
1,1873-03-08,1873,3
2,1874-03-07,1874,3
3,1875-03-06,1875,3
4,1876-03-04,1876,3


In [20]:
df_features["home_advantage"] = df_features["neutral"]

In [21]:
mapping ={"Home Win": 0, "Away Win": 2, "Draw": 1}

df_features["result_encoded"] = df_features["result"].map(mapping)

df_features[["result", "result_encoded"]].head()

,result,result_encoded
0,Draw,1
1,Home Win,0
2,Home Win,0
3,Draw,1
4,Home Win,0


In [22]:
df_features["tournament"].nunique()
df_features["tournament"].value_counts().head(20)

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1012
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

In [23]:
df_features = df_features.sort_values(by="date").reset_index(drop=True)

In [43]:
team_stats={}

def get_team_stats(team):
    if team not in team_stats:
        team_stats[team] = {
            "matches": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_scored": 0,
            "goals_conceded": 0,
            "recent_points":[]
        }
    
    return team_stats[team]

In [44]:
def update_team_stats(stats,goals_for,goals_against):
    stats["matches"] += 1
    stats["goals_scored"] += goals_for
    stats["goals_conceded"] += goals_against
    
    if goals_for > goals_against:
        stats["wins"] += 1
        stats["recent_points"].append(3)
    elif goals_for < goals_against:
        stats["losses"] += 1
        stats["recent_points"].append(0)
    else:
        stats["draws"] += 1
        stats["recent_points"].append(1)

    if len(stats["recent_points"]) > 5:
        stats["recent_points"].pop(0)

In [45]:
feature_columns = [
    "home_matches",
    "away_matches",
    "home_win_rate",
    "away_win_rate",
    "home_avg_goals",
    "away_avg_goals",
    "home_avg_conceded",
    "away_avg_conceded"
]

for col in feature_columns:
    df_features[col] = 0.0

In [46]:
df_features["home_recent_form"]=0.0
df_features["away_recent_form"]=0.0

def calculate_recent_form(stats):
    if len(stats["recent_points"])==0:
        return 0
    return sum(stats["recent_points"])/(len(stats["recent_points"]))

df_features["home_goal_diff"]=0.0
df_features["away_goal_diff"]=0.0

def goal_difference(stats):
   if stats["matches"]==0:
       return 0
   return (stats["goals_scored"]-stats["goals_conceded"])/(stats["matches"])
    


In [47]:
tournament_importance = {

    "Friendly": 1,

    "FIFA World Cup qualification": 3,
    "UEFA Euro qualification": 3,
    "African Cup of Nations qualification": 3,

    "FIFA World Cup": 5,

    "UEFA Euro": 4,
    "Copa América": 4,
    "AFC Asian Cup": 4,
    "African Cup of Nations": 4,
    "CONCACAF Gold Cup": 4
}

df_features["tournament_importance"] = 2

In [48]:
for idx, row in df_features.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    home_stats = get_team_stats(home)
    away_stats = get_team_stats(away)

    df_features.at[idx, "home_matches"] = home_stats["matches"]
    df_features.at[idx, "away_matches"] = away_stats["matches"]

    if home_stats["matches"]>0:
        home_win_rate = home_stats["wins"]/home_stats["matches"]
    else:
        home_win_rate = 0.0

    df_features.at[idx, "home_win_rate"] = home_win_rate

    if away_stats["matches"]>0:
        away_win_rate = away_stats["wins"]/away_stats["matches"]
    else:
        away_win_rate = 0.0

    df_features.at[idx, "away_win_rate"] = away_win_rate

    #Avergae goals scored

    if home_stats["matches"]>0:
        avg = home_stats["goals_scored"]/home_stats["matches"]
    else:
        avg=0

    df_features.at[idx, "home_avg_goals"] = avg

    if away_stats["matches"]>0:
        avg = away_stats["goals_scored"]/away_stats["matches"]
    else:
        avg=0
    df_features.at[idx, "away_avg_goals"] = avg


    #Average goals conceded
    if home_stats["matches"]>0:
        avg = home_stats["goals_conceded"]/home_stats["matches"]
    else:
        avg=0

    df_features.at[idx, "home_avg_goals_conceded"] = avg

    if away_stats["matches"]>0:
        avg = away_stats["goals_conceded"]/away_stats["matches"]
    else:
        avg=0
    df_features.at[idx, "away_avg_goals_conceded"] = avg

    df_features.at[idx, "home_recent_form"] = calculate_recent_form(home_stats)
    df_features.at[idx, "away_recent_form"] = calculate_recent_form(away_stats)

    df_features.at[idx, "home_goal_diff"] = goal_difference(home_stats)
    df_features.at[idx, "away_goal_diff"] = goal_difference(away_stats)

    tournament = row["tournament"]
    importance = tournament_importance.get(tournament, 2)
    df_features.at[idx, "tournament_importance"] = importance


    update_team_stats(home_stats, row["home_score"], row["away_score"])
    update_team_stats(away_stats, row["away_score"], row["home_score"])


In [49]:
df_features[
[
"home_team",
"away_team",
"home_matches",
"away_matches",
"home_win_rate",
"away_win_rate",
"home_avg_goals",
"away_avg_goals",
"home_avg_goals_conceded",
"away_avg_goals_conceded",
"home_recent_form",
"away_recent_form",
"home_goal_diff",
"away_goal_diff",
"tournament_importance",
]
].head(15)

,home_team,away_team,home_matches,away_matches,home_win_rate,away_win_rate,home_avg_goals,away_avg_goals,home_avg_goals_conceded,away_avg_goals_conceded,home_recent_form,away_recent_form,home_goal_diff,away_goal_diff,tournament_importance
0,Scotland,England,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
1,England,Scotland,1.0,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1
2,Scotland,England,2.0,2.0,0.000000,0.500000,1.000000,2.000000,2.000000,1.000000,0.500000,2.000000,-1.000000,1.000000,1
3,England,Scotland,3.0,3.0,0.333333,0.333333,1.666667,1.333333,1.333333,1.666667,1.333333,1.333333,0.333333,-0.333333,1
4,Scotland,England,4.0,4.0,0.250000,0.250000,1.500000,1.750000,1.750000,1.500000,1.250000,1.250000,-0.250000,0.250000,1
5,Scotland,Wales,5.0,0.0,0.400000,0.000000,1.800000,0.000000,1.400000,0.000000,1.600000,0.000000,0.400000,0.000000,1
6,England,Scotland,5.0,6.0,0.200000,0.500000,1.400000,2.166667,1.800000,1.166667,1.000000,2.000000,-0.400000,1.000000,1
7,Wales,Scotland,1.0,7.0,0.000000,0.571429,0.000000,2.285714,4.000000,1.142857,0.000000,2.600000,-4.000000,1.142857,1
8,Scotland,England,8.0,6.0,0.625000,0.166667,2.250000,1.333333,1.000000,2.000000,2.600000,0.800000,1.250000,-0.666667,1
9,Scotland,Wales,9.0,2.0,0.666667,0.000000,2.777778,0.000000,1.111111,3.000000,3.000000,0.000000,1.666667,-3.000000,1


In [50]:
df_clean.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,Draw
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,Home Win
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,Home Win
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,Draw
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,Home Win


In [51]:
df_features.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,away_avg_conceded,home_avg_goals_conceded,away_avg_goals_conceded,home_recent_form,away_recent_form,home_goal_diff,away_goal_diff,tournament_importance,home_h2h_winrate,away_h2h_winrate
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,Draw,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,Home Win,...,0.0,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1,0.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,Home Win,...,0.0,2.000000,1.000000,0.500000,2.000000,-1.000000,1.000000,1,0.0,0.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,Draw,...,0.0,1.333333,1.666667,1.333333,1.333333,0.333333,-0.333333,1,0.0,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,Home Win,...,0.0,1.750000,1.500000,1.250000,1.250000,-0.250000,0.250000,1,0.0,0.0


In [55]:
features = [
    "home_team",
    "away_team",
    "neutral",
    "tournament_importance",

    "home_matches",
    "away_matches",

    "home_win_rate",
    "away_win_rate",

    "home_avg_goals",
    "away_avg_goals",

    "home_avg_goals_conceded",
    "away_avg_goals_conceded",

    "home_recent_form",
    "away_recent_form",

    "home_goal_diff",
    "away_goal_diff"
]

X = df_features[features].copy()
y = df_features["result"]

In [56]:
from sklearn.preprocessing import LabelEncoder

home_encoder = LabelEncoder()
away_encoder = LabelEncoder()
target_encoder = LabelEncoder()

X["home_team"] = home_encoder.fit_transform(X["home_team"])
X["away_team"] = away_encoder.fit_transform(X["away_team"])
y = target_encoder.fit_transform(y)

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [58]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42,n_jobs=-1)

model.fit(X_train, y_train)


,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

In [59]:
predictions = model.predict(X_test)

In [60]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print(accuracy)

0.5473662925892225


In [61]:
#detailed report
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions, target_names=target_encoder.classes_))

              precision    recall  f1-score   support

    Away Win       0.54      0.47      0.50      2872
        Draw       0.27      0.10      0.14      2305
    Home Win       0.59      0.81      0.68      4714

    accuracy                           0.55      9891
   macro avg       0.47      0.46      0.44      9891
weighted avg       0.50      0.55      0.50      9891



In [62]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, predictions)
print(cm)

[[1357  293 1222]
 [ 584  226 1495]
 [ 570  313 3831]]


In [64]:
#feature importance
importances = model.feature_importances_
for feature, score in zip(features, importances):
    print(feature, score)

home_team 0.04864253359369928
away_team 0.048660478614876164
neutral 0.010599836658372967
tournament_importance 0.02143785709350443
home_matches 0.07186576491862376
away_matches 0.07624284670612776
home_win_rate 0.08071880589475931
away_win_rate 0.08273003706104244
home_avg_goals 0.07010829114759223
away_avg_goals 0.07128083732146719
home_avg_goals_conceded 0.07802809107238164
away_avg_goals_conceded 0.0779765388379202
home_recent_form 0.049223332224399056
away_recent_form 0.04737847793478199
home_goal_diff 0.08326607716146178
away_goal_diff 0.08184019375898983


In [65]:
import joblib

joblib.dump(model,"world_cup_predictor.pkl")
joblib.dump(home_encoder,"home_encoder.pkl")
joblib.dump(away_encoder,"away_encoder.pkl")
joblib.dump(target_encoder,"target_encoder.pkl")

['target_encoder.pkl']

In [66]:
from sklearn.preprocessing import LabelEncoder
team_encoder = LabelEncoder()

all_teams = pd.concat([df_features["home_team"], df_features["away_team"]])

team_encoder.fit(all_teams)

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](336,)","['Abkhazia','Afghanistan','Albania',...,'Zanzibar','Zimbabwe', 'Åland Islands']"


In [67]:
X = df_features[features].copy()

X["home_team"] = team_encoder.transform(X["home_team"])
X["away_team"] = team_encoder.transform(X["away_team"])

In [68]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(df_features["result"])

In [69]:

from sklearn.preprocessing import LabelEncoder

X = df_features[features].copy()
y = df_features["result"]

team_encoder = LabelEncoder()

import pandas as pd

all_teams = pd.concat([
    df_features["home_team"],
    df_features["away_team"]
])

team_encoder.fit(all_teams)

#encoding the home and away teams using the fitted LabelEncoder
X["home_team"] = team_encoder.transform(X["home_team"])
X["away_team"] = team_encoder.transform(X["away_team"])

#encoding the target variable (result) using LabelEncoder
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [77]:
predictions = model.predict(X_test)

In [79]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    target_names=target_encoder.classes_
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.556869881710646

Classification Report:
              precision    recall  f1-score   support

    Away Win       0.56      0.47      0.51      2872
        Draw       0.27      0.08      0.12      2305
    Home Win       0.58      0.84      0.69      4714

    accuracy                           0.56      9891
   macro avg       0.47      0.46      0.44      9891
weighted avg       0.50      0.56      0.51      9891


Confusion Matrix:
[[1346  242 1284]
 [ 553  182 1570]
 [ 494  240 3980]]


In [80]:
importance = model.feature_importances_

print("\nFeature Importance:")

for feature, score in zip(features, importance):
    print(f"{feature}: {score:.4f}")


Feature Importance:
home_team: 0.0487
away_team: 0.0484
neutral: 0.0108
tournament_importance: 0.0212
home_matches: 0.0709
away_matches: 0.0750
home_win_rate: 0.0801
away_win_rate: 0.0834
home_avg_goals: 0.0691
away_avg_goals: 0.0708
home_avg_goals_conceded: 0.0785
away_avg_goals_conceded: 0.0784
home_recent_form: 0.0472
away_recent_form: 0.0464
home_goal_diff: 0.0858
away_goal_diff: 0.0854


In [81]:
import joblib

joblib.dump(model, "../models/world_cup_predictor.pkl")
joblib.dump(team_encoder, "../models/team_encoder.pkl")
joblib.dump(target_encoder, "../models/target_encoder.pkl")

['../models/target_encoder.pkl']

In [82]:
loaded_model = joblib.load("../models/world_cup_predictor.pkl")
loaded_team_encoder = joblib.load("../models/team_encoder.pkl")
loaded_target_encoder = joblib.load("../models/target_encoder.pkl")

print("Model loaded successfully!")
print("Number of teams:", len(loaded_team_encoder.classes_))
print("Classes:", loaded_target_encoder.classes_)

Model loaded successfully!
Number of teams: 336
Classes: ['Away Win' 'Draw' 'Home Win']
